<a href="https://colab.research.google.com/github/Ali-Hamza-developer/NLP/blob/main/01_regex_for_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 align='center'>NLP Tutorial: Regular Expressions (Regex)</h1>

### Kya seekhenge is notebook mein?
- Regex ka use customer support chats se info nikalne ke liye
- Wikipedia jaisi text se structured info (name, age, birth date, birth place) extract karna
- Extra regex concepts jo NLP preprocessing mein kaam aate hain (hashtags, mentions, URLs, dates, currency, stopword-jaisi cleanup)
- Har code cell ke sath comments taake line-by-line samajh aaye

<h3>(1) Regex basics — order number nikalna (customer support chat)</h3>

In [1]:
import re

# Sample chat 1: order number likha hai "order # 412889912"
chat1 = 'Hello, I am having an issue with my order # 412889912'

# Pattern samajh: 'order' ke baad jo bhi non-digit chars hain unko skip karo ([^\d]*),
# fir jo digits milein wo capture karo (\d*)
pattern = r'order[^\d]*(\d*)'
matches = re.findall(pattern, chat1)
matches  # -> ['412889912']

['412889912']

In [2]:
# Sample chat 2: thodi different wording "order number 412889912"
chat2 = 'I have a problem with my order number 412889912'
pattern = r'order[^\d]*(\d*)'
matches = re.findall(pattern, chat2)
matches

['412889912']

In [3]:
# Sample chat 3: order number ke sath extra info (price complaint) bhi hai
chat3 = 'My order 412889912 is having an issue, I was charged 300$ when online it says 280$'
pattern = r'order[^\d]*(\d*)'
matches = re.findall(pattern, chat3)
matches

['412889912']

**Helper function** — baar baar `re.findall` likhne ke bajaye ek reusable function bana lete hain.
Ye function pattern match karega aur agar match mile to sirf pehla match return karega.

In [4]:
def get_pattern_match(pattern, text):
    """Given a regex pattern aur text, pehla match return karta hai (agar mila to)."""
    matches = re.findall(pattern, text)
    if matches:
        return matches[0]
    return None  # koi match nahi mila

In [5]:
# ab helper function use karke order number nikalte hain
get_pattern_match(r'order[^\d]*(\d*)', chat1)

'412889912'

<h3>Email ID aur Phone Number nikalna</h3>

In [6]:
# 3 alag chats — har ek mein email/phone likhne ka style different hai
chat1 = 'you ask lot of questions 😠  1235678912, abc@xyz.com'
chat2 = 'here it is: (123)-567-8912, abc@xyz.com'
chat3 = 'yes, phone: 1235678912 email: abc@xyz.com'

**-----Email ID-----**

In [7]:
# Email pattern: letters/numbers/underscore, @, letters, dot, letters/numbers
email_pattern = r'[a-zA-Z0-9_]*@[a-z]*\.[a-zA-Z0-9]*'
get_pattern_match(email_pattern, chat1)

'abc@xyz.com'

In [8]:
get_pattern_match(email_pattern, chat2)

'abc@xyz.com'

In [9]:
get_pattern_match(email_pattern, chat3)

'abc@xyz.com'

**-----Phone number-----**

In [10]:
# Do formats handle kar rahe hain: plain 10-digit number, ya (123)-456-7890 wala format
phone_pattern = r'(\d{10})|(\(\d{3}\)-\d{3}-\d{4})'
get_pattern_match(phone_pattern, chat1)   # plain 10 digit number milega

('1235678912', '')

In [11]:
get_pattern_match(phone_pattern, chat2)   # bracket format milega, tuple mein 2nd position par

('', '(123)-567-8912')

In [12]:
get_pattern_match(phone_pattern, chat3)   # plain 10 digit number, tuple mein 1st position par

('1235678912', '')

<h3>(2) Regex for Information Extraction — Wikipedia-style text</h3>

Ab hum ek bada text block lenge (jaisa Wikipedia infobox mein hota hai) aur usse
**name, age, birth date, birth place** nikalenge regex se.

In [13]:
text = '''
Born\tElon Reeve Musk
June 28, 1971 (age 50)
Pretoria, Transvaal, South Africa
Citizenship\t
South Africa (1971–present)
Canada (1971–present)
United States (2002–present)
Education\tUniversity of Pennsylvania (BS, BA)
Title\t
Founder, CEO and Chief Engineer of SpaceX
CEO and product architect of Tesla, Inc.
Founder of The Boring Company and X.com (now part of PayPal)
Co-founder of Neuralink, OpenAI, and Zip2
Spouse(s)\t
Justine Wilson
​
​(m. 2000; div. 2008)​
Talulah Riley
​
​(m. 2010; div. 2012)​
​
​(m. 2013; div. 2016)
'''

In [14]:
# Age nikalna: "age" ke baad digits dhundo
get_pattern_match(r'age (\d+)', text)

'50'

In [15]:
# Name nikalna: "Born" ke baad, next newline tak jo bhi hai wo naam hai
get_pattern_match(r'Born(.*)\n', text).strip()

'Elon Reeve Musk'

In [16]:
# Birth date: "Born" wali line ke agli line mein, "(age" se pehle tak
get_pattern_match(r'Born.*\n(.*)\(age', text).strip()

'June 28, 1971'

In [17]:
# Birth place: "(age...)" wali line ke turant baad wali line
get_pattern_match(r'\(age.*\n(.*)', text)

'Pretoria, Transvaal, South Africa'

**Sab kuch ek function mein combine kar dete hain — reusable ban jayega har person ke liye.**

In [18]:
def extract_personal_information(text):
    """Wikipedia-style infobox text se age, name, birth_date, birth_place nikalta hai."""
    age = get_pattern_match(r'age (\d+)', text)
    full_name = get_pattern_match(r'Born(.*)\n', text)
    birth_date = get_pattern_match(r'Born.*\n(.*)\(age', text)
    birth_place = get_pattern_match(r'\(age.*\n(.*)', text)

    return {
        'age': int(age) if age else None,
        'name': full_name.strip() if full_name else None,
        'birth_date': birth_date.strip() if birth_date else None,
        'birth_place': birth_place.strip() if birth_place else None
    }

In [19]:
extract_personal_information(text)

{'age': 50,
 'name': 'Elon Reeve Musk',
 'birth_date': 'June 28, 1971',
 'birth_place': 'Pretoria, Transvaal, South Africa'}

In [20]:
# Dusra example — kisi aur person ka text test karte hain
text = '''
Born\tMukesh Dhirubhai Ambani
19 April 1957 (age 64)
Aden, Colony of Aden
(present-day Yemen)[1][2]
Nationality\tIndian
Alma mater\t
St. Xavier's College, Mumbai
Institute of Chemical Technology (B.E.)
Stanford University (drop-out)
Occupation\tChairman and MD, Reliance Industries
Spouse(s)\tNita Ambani ​(m. 1985)​[3]
Children\t3
Parent(s)\t
Dhirubhai Ambani (father)
Kokilaben Ambani (mother)
Relatives\tAnil Ambani (brother)
Tina Ambani (sister-in-law)
'''

In [21]:
extract_personal_information(text)

{'age': 64,
 'name': 'Mukesh Dhirubhai Ambani',
 'birth_date': '19 April 1957',
 'birth_place': 'Aden, Colony of Aden'}

<h3>(3) Extra Regex Concepts jo NLP preprocessing mein bohot use hote hain</h3>

Ye chhote chhote practical examples hain — har ek NLP text-cleaning pipeline ka common step hota hai.

**-----Hashtags aur Mentions (Twitter/Social media text)-----**

In [22]:
tweet = 'Loving the new #MachineLearning course by @codeacademy! #NLP #AI 🔥'

# Hashtags: '#' ke baad word characters
hashtags = re.findall(r'#\w+', tweet)
hashtags

['#MachineLearning', '#NLP', '#AI']

In [23]:
# Mentions: '@' ke baad word characters
mentions = re.findall(r'@\w+', tweet)
mentions

['@codeacademy']

**-----URLs nikalna aur text se remove karna-----**

In [24]:
text_with_url = 'Check the docs here https://example.com/docs/regex and read more.'

# URL pattern: http/https se start, phir non-space characters
url_pattern = r'https?://\S+'
urls = re.findall(url_pattern, text_with_url)
urls

['https://example.com/docs/regex']

In [25]:
# Text se URL remove karna (NLP cleaning step)
clean_text = re.sub(url_pattern, '', text_with_url).strip()
clean_text

'Check the docs here  and read more.'

**-----Dates ka generic pattern-----**

In [26]:
date_text = 'The meeting is scheduled on 12/08/2025 and follow-up on 2026-01-15.'

# dd/mm/yyyy format
dates_slash = re.findall(r'\d{2}/\d{2}/\d{4}', date_text)
dates_slash

['12/08/2025']

In [27]:
# yyyy-mm-dd format
dates_dash = re.findall(r'\d{4}-\d{2}-\d{2}', date_text)
dates_dash

['2026-01-15']

**-----Currency / Price extraction-----**

In [28]:
price_text = 'The laptop costs $1200 but earlier it was priced at $999.'

# $ ke baad digits (comma ya decimal ke saath bhi handle ho sakta hai)
prices = re.findall(r'\$\d+(?:,\d{3})*(?:\.\d+)?', price_text)
prices

['$1200', '$999']

**-----Stopword-style noise removal (punctuation + extra spaces)-----**

In [29]:
noisy_text = "Hello!!!   this   is  ...  a messy,,, NLP text???"

# Extra punctuation hatana
step1 = re.sub(r'[!?.,]+', ' ', noisy_text)

# Multiple spaces ko single space mein convert karna
clean = re.sub(r'\s+', ' ', step1).strip()
clean

'Hello this is a messy NLP text'

**-----Word tokenization (basic, regex-based)-----**

In [30]:
sentence = "Regex-based tokenization isn't perfect, but it's fast!"

# Simple word tokenizer: letters/numbers/apostrophe ko ek token maano
tokens = re.findall(r"[A-Za-z0-9']+", sentence)
tokens

['Regex', 'based', 'tokenization', "isn't", 'perfect', 'but', "it's", 'fast']

### Summary
- `re.findall` — sab matches ek list mein deta hai
- `re.sub` — pattern ko replace/remove karne ke liye
- Groups `(...)` — specific part capture karne ke liye
- `[^\d]` jaisa negated class — "in nahi chahiye" wale chars skip karne ke liye
- Ye sab NLP preprocessing (cleaning, tokenizing, entity extraction) ka foundation hai

<h3>Practice / Exercise</h3>

Apne khud ke examples try karo:
- Kisi bhi paragraph se saare numbers nikalo
- Kisi email signature se phone + email dono ek sath nikalo
- Kisi tweet se saare hashtags count karo